<a href="https://colab.research.google.com/github/Ali-Hamza-developer/NLP/blob/main/04_spacy_pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# spaCy Language Processing Pipelines

Easy explanation + code, both files combined in one place. Minimal diagrams, maximum clarity.


## 1. What is a "Pipeline" in spaCy?

When you pass a text to `nlp(text)`, spaCy doesn't do everything in one shot. It passes your text through a **series of steps (components)**, one after another. That series of steps is called the **pipeline**.

```
text → tokenizer → tagger → parser → ner → attribute_ruler → lemmatizer → doc (final result)
```

- **Tokenizer** always runs first — it just splits text into tokens (words/punctuation). Every pipeline has this, even a blank one.
- Everything after tokenizer (tagger, parser, ner, lemmatizer...) is **optional** and only exists if you load a **trained model** (like `en_core_web_sm`).

That's it — no need for a big diagram, just remember: **blank pipeline = tokenizer only. Trained pipeline = tokenizer + extra smart components.**

## 2. Blank Pipeline — Only Tokenizer, No Intelligence

`spacy.blank("en")` gives you a pipeline with **nothing but a tokenizer**. It can split text into tokens, but it has **no idea** about grammar, entities, etc.

In [1]:
import spacy

nlp = spacy.blank("en")   # blank pipeline -> only tokenizer, nothing else

doc = nlp("Captain america ate 100$ of samosa. Then he said I can do this all day.")

for token in doc:
    print(token)


Captain
america
ate
100
$
of
samosa
.
Then
he
said
I
can
do
this
all
day
.


**Output:** just the tokens, split nicely. But if you try `token.pos_` or `token.lemma_` here, it will be **empty/useless** because there's no tagger or lemmatizer loaded yet.

In [2]:
nlp.pipe_names   # empty list -> confirms: no components other than tokenizer


[]

## 3. Trained Pipeline — Download & Load (The Useful One)

Download once in terminal:
```
python -m spacy download en_core_web_sm
```
`sm` = small model. Other options: `md` (medium), `lg` (large) — bigger = more accurate but slower/heavier.

Now load it:

In [3]:
nlp = spacy.load("en_core_web_sm")
nlp.pipe_names


['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

**Output:** `['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']`

Now this pipeline knows grammar (POS tags), sentence structure (parser), word roots (lemmatizer), and named entities (ner). Let's use it:

In [4]:
doc = nlp("Captain america ate 100$ of samosa. Then he said I can do this all day.")

for token in doc:
    print(token, " | ", spacy.explain(token.pos_), " | ", token.lemma_)


Captain  |  proper noun  |  Captain
america  |  proper noun  |  america
ate  |  verb  |  eat
100  |  numeral  |  100
$  |  numeral  |  $
of  |  adposition  |  of
samosa  |  proper noun  |  samosa
.  |  punctuation  |  .
Then  |  adverb  |  then
he  |  pronoun  |  he
said  |  verb  |  say
I  |  pronoun  |  I
can  |  auxiliary  |  can
do  |  verb  |  do
this  |  pronoun  |  this
all  |  determiner  |  all
day  |  noun  |  day
.  |  punctuation  |  .


Now every token has a **POS tag** (what kind of word it is) and a **lemma** (base/root form, e.g. "ate" → "eat"). This is the difference a trained pipeline makes vs a blank one.

## 4. Named Entity Recognition (NER) — Find People, Companies, Money, etc.

This uses the `ner` component, which is only available in a trained pipeline.

In [5]:
doc = nlp("Tesla Inc is going to acquire twitter for $45 billion")
for ent in doc.ents:
    print(ent.text, " | ", ent.label_, " | ", spacy.explain(ent.label_))


Tesla Inc  |  ORG  |  Companies, agencies, institutions, etc.
$45 billion  |  MONEY  |  Monetary values, including unit


**Output:**
```
Tesla Inc  |  ORG   |  Companies, agencies, institutions, etc.
$45 billion  |  MONEY |  Monetary values, including unit
```

Optional — visualize entities nicely (only if you're in Jupyter):

In [6]:
from spacy import displacy

displacy.render(doc, style="ent")


## 5. Adding Just ONE Component to a Blank Pipeline (Faster, Lighter)

Sometimes you don't need the full trained pipeline (tagger + parser + lemmatizer + ner) — maybe you **only** want NER, and want it faster/lighter. You can copy just that one component from a trained model into a blank pipeline:

In [7]:
source_nlp = spacy.load("en_core_web_sm")

nlp = spacy.blank("en")
nlp.add_pipe("ner", source=source_nlp)   # only bring the NER component, skip the rest
nlp.pipe_names


['ner']

In [8]:
doc = nlp("Tesla Inc is going to acquire twitter for $45 billion")
for ent in doc.ents:
    print(ent.text, ent.label_)


Tesla Inc ORG
$45 billion MONEY


Same NER result, but this pipeline is lighter because it skipped tagger, parser, and lemmatizer.

## 6. Pipelines for Other Languages

spaCy supports many languages. For French:
```
python -m spacy download fr_core_news_sm
```

In [13]:
# !python -m spacy download fr_core_news_sm

In [12]:
nlp_fr = spacy.load("fr_core_news_sm")

doc = nlp_fr("Tesla Inc va racheter Twitter pour $45 milliards de dollars")
for ent in doc.ents:
    print(ent.text, " | ", ent.label_, " | ", spacy.explain(ent.label_))


Tesla Inc  |  PER  |  Named person or family.
Twitter  |  MISC  |  Miscellaneous entities, e.g. events, nationalities, products or works of art


---
## Quick Cheat Sheet

| Task | Code |
|---|---|
| Blank pipeline (tokenizer only) | `spacy.blank("en")` |
| Trained pipeline (full power) | `spacy.load("en_core_web_sm")` |
| See pipeline components | `nlp.pipe_names` |
| POS tag + lemma | `token.pos_`, `token.lemma_` |
| Named entities | `doc.ents`, `ent.text`, `ent.label_` |
| Add only one component to blank | `nlp.add_pipe("ner", source=trained_nlp)` |
| Visualize entities | `displacy.render(doc, style="ent")` |

---

## Exercise 1 — Get All Proper Nouns + Count

**Task:** From the text below, extract all proper nouns (`PROPN`) into a list, and count them.

In [14]:
nlp = spacy.load("en_core_web_sm")

text = '''Ravi and Raju are the best friends from school days.They wanted to go for a world tour and
visit famous cities like Paris, London, Dubai, Rome etc and also they called their another friend Mohan to take part of this world tour.
They started their journey from Hyderabad and spent next 3 months travelling all the wonderful cities in the world and cherish a happy moments!
'''

doc = nlp(text)

proper_nouns = []

for token in doc:
    if token.pos_ == "PROPN":
        proper_nouns.append(token)

print("Proper Nouns: ", proper_nouns)
print("Count: ", len(proper_nouns))


Proper Nouns:  [Raju, Paris, London, Dubai, Rome, Mohan, Hyderabad]
Count:  7


## Exercise 2 — Get All Company Names + Count

**Task:** Use NER (`ORG` label) to pull out company names from the text.

In [15]:
text = '''The Top 5 companies in USA are Tesla, Walmart, Amazon, Microsoft, Google and the top 5 companies in
India are Infosys, Reliance, HDFC Bank, Hindustan Unilever and Bharti Airtel'''

doc = nlp(text)

companies = []

for ent in doc.ents:
    if ent.label_ == "ORG":
        companies.append(ent)

print("Company Names: ", companies)
print("Count: ", len(companies))


Company Names:  [Tesla, Walmart, Amazon, Microsoft, Google, Infosys, Reliance, HDFC Bank, Hindustan Unilever, Bharti Airtel]
Count:  10
